##### 我来详细介绍 verl 的 NCCL 重分片 机制，这是 3D-HybridEngine 中最核心的技术实现。

# NCCL 重分片要解决的问题
## 核心矛盾：<font color='red'>同一模型，两种并行布局</font>

In [ ]:
Actor 模型在训练时 (Megatron-LM)          Actor 模型在生成时 (vLLM)
┌─────────────────────────────┐          ┌─────────────────────────────┐
│  TP=8, PP=4, DP=2           │          │  TP=4, PP=1, DP=16          │
│  (64 GPUs)                  │          │  (64 GPUs)                  │
│                             │          │                             │
│  GPU 0-7:   Layer 0-15 的   │          │  GPU 0-3:   Layer 0-63 的   │
│             1/8 列分片       │   ???    │             1/4 列分片       │
│  GPU 8-15:  Layer 0-15 的   │  ──────→ │  GPU 4-7:   Layer 0-63 的   │
│             1/8 列分片       │  如何    │             1/4 列分片       │
│  ...                        │  转换?   │  ...                        │
│  GPU 16-23: Layer 16-31 的  │          │                             │
│             1/8 列分片       │          │  (无流水线，所有层完整)       │
└─────────────────────────────┘          └─────────────────────────────┘

## 问题本质：
- Megatron 用 TP + PP + DP 切分模型，每层只存部分参数
- vLLM 用 TP + DP（<font color='red'>通常 PP=1</font>），需要每层有完整参数（按 TP 切列）
- 两者之间不是简单的 "全量聚合再重新分片"，而是需要在分布式环境下 局部、高效、原地 完成转换

### 为什么推理阶段通常会配置较小的 PP？
- 这主要是由推理阶段的物理特性决定的。
- 推理（生成）过程对延迟（Latency）更为敏感。<font color='red'>如果采用较大的 PP，会导致模型在不同设备间频繁进行流水线通信，从而增加推理延迟。</font>
- 因此，在推理阶段，系统通常会配置较小的 PP（以及较小的 TP），以换取更快的生成速度。

# NCCL 重分片的核心思想
verl 的关键洞察：不需要在全局 64 GPU 间做全量 AllGather，而是<font color='red'>利用 micro DP 组的局部性</font>。
## micro DP 组的定义

在 veRL（HybridFlow）框架中，<font color='red'>为了解决 Actor 模型在训练（Training）和生成（Generation）阶段因并行策略不同而导致的参数重分片（Resharding）通信开销问题，引入了 micro DP（微数据并行） 机制。</font>

##### 以下是关于 micro DP 及其在 NCCL 重分片中作用的详细解释：

### 1. 为什么需要 micro DP？（背景与痛点）
- 在 RLHF 训练中，
  - Actor 模型在训练阶段通常需要较大的模型并行度（如 TP=2, DP=2）来分摊显存和计算压力；
  - <font color='blue'>而在生成阶段，由于是纯推理且计算密度低，通常会切换为较小的模型并行度（如 TP=1, DP=4）以提高资源利用率。</font>
  
- <font color='red'>当 Actor 从训练阶段切换到生成阶段时，需要将模型参数从训练时的并行组重新分发（Resharding）到生成时的并行组。</font>如果直接在整个 DP 组内进行全局的 all-gather 操作来收集全量参数，会导致极大的通信开销。

### 2. 什么是 micro DP？
- micro DP（微数据并行） 是将原本较大的数据并行（DP）组进一步拆分为更小的子组（即 micro DP group）。veRL 的 3D-HybridEngine <font color='red'>正是以这些 micro DP group 为基本单位来进行参数的收集与重分片的。</font>

### 3. micro DP 在重分片中的具体工作流程
在 RLHF 的第 i+1 次迭代中，3D-HybridEngine 利用 micro DP 进行参数重分片的具体步骤如下：
1. <font color='red'>组内 All-Gather</font>：
  - 首先，在每个 micro DP group 内部执行 all-gather 操作。这样，每个 micro DP 组内的设备都能收集到该组所负责的那部分 Actor 模型参数。
2. <font color='red'>跨组组合</font>：
  - 接着，将不同 micro DP group 中收集到的参数进行组合拼接。
3. <font color='red'>按新策略重分片</font>：
  - 最后，根据 Actor 训练阶段设定的 3D 并行策略，将组合后的完整参数重新分片（Reshard）并分发给各个设备，准备进入下一轮训练。
  
## 场景设定
- 假设我们有一个 Actor 模型，<font color='red'>当前处于 生成（Generation）阶段</font>，使用的并行策略为 <font color='blue'>TP=4, DP=2</font>
  - （即 4 张 GPU 负责张量并行，2 组 GPU 负责数据并行）。

- 现在，<font color='red'>生成阶段结束，准备切换到 训练（Training）阶段</font>，训练阶段设定的并行策略为 <font color='blue'>TP=2, DP=4</font>
  - （即 2 张 GPU 负责张量并行，4 组 GPU 负责数据并行）。

##### 在这个切换过程中，veRL 引入了 micro DP 机制来进行参数重分片。具体步骤如下：

### 第一步：在 micro DP 组内进行 All-Gather
veRL 并没有直接让所有 GPU 参与全局的参数收集，<font color='red'>而是将原本的 DP=2 进一步拆分，构建了更小的 micro_dp_group。</font>
- 在上述 TP=4, DP=2 的配置下，<font color='red'>veRL 会将属于同一个 micro_dp_group 的 4 张 GPU 的参数进行 all-gather 并拼接（concat）起来。</font>
- <font color='blue'>这样，每个 micro_dp_group 内部就拥有了满足 TP=2 所需的那部分模型参数。</font>

### 第二步：数据层面的 Chunk 与 Gather 配合
在参数重分片的同时，数据也需要重新切分。
- 系统会将原始的输入数据根据 micro_dp_group 进行切分（chunk）。
- 当生成任务完成后，再将这些数据 gather 并 concat 起来，以匹配新的并行策略。

### 第三步：按新策略重分片（TP=4 变 TP=2）
- 收集并拼接好参数后，系统会将 TP=4 的配置转换为 TP=2。
- 此时，每张 GPU 只需要保留自己在新 TP=2 策略下负责的那部分参数，主动抛弃（释放）掉自己不需要的参数部分，从而顺利进入训练阶段。

### 总结
通过这个例子可以看出，veRL 在并行切换时有一套清晰的数据处理套路：DP 维度减小，就要 gather；DP 维度增加，就要 chunk。
借助 micro DP 机制，veRL 避免了在切换时进行全局的全量参数通信，不仅实现了零内存冗余，还最大程度地降低了通信开销。

## 场景设定（与上一轮完全对称）
- 起点（训练阶段）：TP=2, DP=4（8张卡，4个DP组，每组2张卡做张量并行）
- 终点（生成阶段）：TP=4, DP=2（8张卡，2个DP组，每组4张卡做张量并行）

##### 反向推演：从训练阶段（TP=2, DP=4）切换到生成阶段（TP=4, DP=2）

### 1. 痛点分析：为什么不能直接 All-Gather？
在训练阶段，每张卡只拥有 1/2 的模型参数（因为 TP=2）。如果直接为了生成阶段（TP=4）而进行全局 All-Gather，收集完整的模型参数，不仅需要巨大的通信带宽，还需要每张卡额外分配 1/2 的显存来存放这些冗余参数。这在显存极其紧张的 RLHF 训练中是不可接受的。

### 2. micro DP 的介入：精准“切蛋糕”
veRL 巧妙地利用 micro DP 将原本庞大的 DP=4 拆分为两个更小的 micro DP 组（每个组包含 4 张卡）。
- 操作：系统不再收集全量参数，而是只在每个 micro DP 组内部进行 All-Gather。
- 结果：每个 micro DP 组内的 4 张卡，通过组内通信，刚好拼凑出了生成阶段（TP=4）所需的完整模型参数。

### 3. 数据层面的反向操作：Chunk（切分）
与上一轮例子（DP 减小需要 Gather）相反，这次 DP 维度从 4 增加到了 2，数据需要重新切分。
- 操作：系统会将原本在训练阶段（DP=4）切分为 4 份的数据，根据新的 micro DP 组进行重新组合（Gather & Concat），然后按照生成阶段（DP=2）的策略，重新切分（Chunk）为 2 份。

### 4. 零冗余过渡
通过这种组内 All-Gather 的方式，每张卡只保留了生成阶段自己需要的那 1/4 模型参数，主动释放了训练阶段多余的参数副本，顺利进入生成阶段。

### 核心总结：veRL 的“动态伸缩”哲学
通过正反两个例子的对比，我们可以提炼出 veRL 3D-HybridEngine 在并行切换时的底层逻辑：
1. DP 维度减小（如 DP=4 变 DP=2）：需要 Gather（聚合）。micro DP 在组内完成聚合，避免全局通信。
2. DP 维度增大（如 DP=2 变 DP=4）：需要 Chunk（切分）。micro DP 帮助系统精准定位需要切分和重组的数据块。

##### 一句话概括：
- micro DP 就像是一个“通信缓冲区”，它把原本需要在 8 张卡之间进行的“全局大搬运”，降级成了 4 张卡之间的“局部小搬运”。无论 DP 维度是变大还是变小，它都能确保参数和数据以最小的通信开销和零显存冗余完成无缝切换。

## DP 变成 micro DP 的具体过程

在 veRL 的 3D-HybridEngine 中，micro DP 并不是一个固定的概念，而是为了适应不同并行策略切换而动态构建的子组。<font color='red'>它的切分方式完全取决于训练和生成阶段的并行度差异。</font>

##### 场景设定
<font color='red'>假设我们有 8 张 GPU：</font>
- <font color='blue'>训练阶段（Training）：采用 TP=2, DP=4。</font>
  - 这 8 张卡被切分为 4 个 DP 组，每个组包含 2 张卡（例如：[GPU0, GPU1]、[GPU2, GPU3]、[GPU4, GPU5]、[GPU6, GPU7]）。
- <font color='blue'>生成阶段（Generation）：采用 TP=4, DP=2。</font>
  - 这 8 张卡被切分为 2 个 DP 组，每个组包含 4 张卡（例如：[GPU0, GPU1, GPU2, GPU3]、[GPU4, GPU5, GPU6, GPU7]）。

##### DP 变成 micro DP 的具体过程
- 当模型从“训练阶段”切换到“生成阶段”时，由于 TP 从 2 变成了 4，DP 从 4 变成了 2，<font color='red'>系统需要重新分配模型参数。</font>此时，veRL 会基于生成阶段的配置，重新构建 micro DP 组：
### 第一步：改变切分优先级，构建 micro DP 组
veRL 改进了切分优先级，不再使用默认的切分规则，而是专门针对这种并行度转换构建了 MICRO_DATA_PARALLEL_GROUP。
- <font color='red'>系统会将原本属于训练阶段 DP=4 的 4 个小分组“打散”，然后按照生成阶段 DP=2 的需求，重新“聚拢”成 2 个更大的分组。</font>
- 结果：原来的 4 个 DP 组，变成了 2 个 micro DP 组。
  - micro DP Group 1：[GPU0, GPU1, GPU2, GPU3]
  - micro DP Group 2：[GPU4, GPU5, GPU6, GPU7]
  
### 第二步：组内参数重组（Gather & Concat）
<font color='blue'>在生成阶段，每张卡只需要 1/4 的模型参数（因为 TP=4）。</font>
- <font color='red'>系统会将属于同一个 micro DP group 的参数进行 gather 并拼接（concat）起来。</font>
- 这样，每个 micro DP 组内的 4 张卡，刚好凑齐了满足 TP=4 时所需的那部分模型参数。

### 第三步：数据层面的配合切分
参数重组的同时，数据也要跟着变：
- 系统会将原始的输入数据根据这两个新的 micro DP group 进行切分（chunk）。
- 生成任务结束后，再将这些数据 gather 并 concat 起来，为切回训练阶段做准备。

### 核心总结
- <font color='blue'>DP 变成 micro DP 的本质，就是根据目标并行策略（比如 TP=4, DP=2）重新划定通信边界。</font>
- veRL 并没有让 8 张卡在全局范围内盲目地互相通信，而是聪明地划分出了 2 个 micro DP 组。所有的参数重组和数据切分，都只在这 4 张卡的小圈子（micro DP 组）内部进行。这种“化整为零”的策略，完美避免了全局通信带来的显存爆炸和带宽瓶颈。

In [ ]:
64 GPU, Megatron 配置: TP=8, PP=4, DP=2

┌─────────────────────────────────────────────────────────────────────┐
│                        64 GPUs 拓扑                                  │
│                                                                     │
│   PP Stage 0 (Layers 0-15)            PP Stage 1 (Layers 16-31)    │
│   ┌─────────┬─────────┐              ┌─────────┬─────────┐         │
│   │  TP组0  │  TP组1  │              │  TP组2  │  TP组3  │         │
│   │ GPU 0-7 │ GPU 8-15│              │GPU 16-23│GPU 24-31│         │
│   │ (DP=0)  │ (DP=0)  │              │ (DP=0)  │ (DP=0)  │         │
│   └─────────┴─────────┘              └─────────┴─────────┘         │
│                                                                     │
│   PP Stage 2 (Layers 32-47)           PP Stage 3 (Layers 48-63)    │
│   ┌─────────┬─────────┐              ┌─────────┬─────────┐         │
│   │  TP组4  │  TP组5  │              │  TP组6  │  TP组7  │         │
│   │GPU 32-39│GPU 40-47│              │GPU 48-55│GPU 56-63│         │
│   │ (DP=1)  │ (DP=1)  │              │ (DP=1)  │ (DP=1)  │         │
│   └─────────┴─────────┘              └─────────┴─────────┘         │
│                                                                     │
│   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━    │
│   micro DP 组 = 跨 PP Stage、同 TP rank、同 DP rank 的 GPU 集合      │
│                                                                     │
│   例如 micro DP 组 0: {GPU 0, GPU 16, GPU 32, GPU 48}               │
│   它们都持有各自 PP Stage 中 TP rank=0 的分片                        │
│   合起来就是完整的一层（按 TP 维度拼接）                              │
└─────────────────────────────────────────────────────────────────────┘

## 关键观察：
- 同一 micro DP 组内的 GPU 持有 不同 PP Stage 的同位置 TP 分片
- 在 micro DP 组内做 all_gather，就能得到 完整的一层（仍按 TP 维度分片）
- 不需要跨所有 64 GPU 通信，只需在 4 GPU 的 micro DP 组内通信


# NCCL 重分片的完整流程
## 阶段 1：PP 维度聚合（micro DP 组内 all_gather）

In [ ]:
目标: 把分散在 4 个 PP Stage 的同位置分片聚合成完整层

micro DP 组 0: {GPU 0, GPU 16, GPU 32, GPU 48}

训练前 (Megatron):
┌─────────┐  ┌─────────┐  ┌─────────┐  ┌─────────┐
│ GPU 0   │  │ GPU 16  │  │ GPU 32  │  │ GPU 48  │
│ Layer   │  │ Layer   │  │ Layer   │  │ Layer   │
│ 0-15    │  │ 16-31   │  │ 32-47   │  │ 48-63   │
│ 的 1/8  │  │ 的 1/8  │  │ 的 1/8  │  │ 的 1/8  │
│ 列分片   │  │ 列分片   │  │ 列分片   │  │ 列分片   │
└─────────┘  └─────────┘  └─────────┘  └─────────┘
     │            │            │            │
     └────────────┴────────────┴────────────┘
                  ↓ NCCL all_gather
┌─────────────────────────────────────────────────┐
│  每个 GPU 获得: Layer 0-63 的 1/8 列分片          │
│  (按 PP 维度拼接: [L0-15|L16-31|L32-47|L48-63])  │
└─────────────────────────────────────────────────┘

### NCCL 通信原语：

In [ ]:
# 创建 micro DP 通信组
micro_dp_group = dist.new_group(ranks=[0, 16, 32, 48])

# 每个 GPU 持有自己的分片: local_shard.shape = [16_layers, hidden, hidden/8]
local_shard = get_local_shard(layer_name)

# all_gather: 收集 micro DP 组内所有分片
# 输出: gathered.shape = [64_layers, hidden, hidden/8]
gathered = torch.empty([64, hidden, hidden//8], device='cuda')

dist.all_gather_into_tensor(gathered, local_shard, group=micro_dp_group)

## 阶段 2：TP 维度重分片（MegatronVLLMShardingManager）
<font color='blue'>PP 聚合后，每层已有完整参数，但 TP 切分方式可能与 vLLM 不兼容。</font>

In [ ]:
Megatron 的 TP 切分方式                vLLM 的 TP 切分方式
     ↓                                      ↓

Q 投影: [d_model × d_head]              Q 投影: [d_model × d_head]
        ↓ 按列切 8 份                            ↓ 按列切 4 份
   [Q0|Q1|Q2|Q3|Q4|Q5|Q6|Q7]           [Q0'|Q1'|Q2'|Q3']
   每份 shape: [d_model, d_head/8]      每份 shape: [d_model, d_head/4]

K 投影: 同理 8 份                        K 投影: 同理 4 份
V 投影: 同理 8 份                        V 投影: 同理 4 份

问题: Megatron 的 8 份 → vLLM 的 4 份，需要重新拼接

### 层名映射与重分片：

In [ ]:
class MegatronVLLMShardingManager:
    def __init__(self, layer_name_mapping):
        """
        layer_name_mapping: Megatron 层名 ↔ vLLM 层名的映射表
        例如:
        {
            'decoder.layers.0.self_attention.query': 'model.layers.0.self_attn.qkv_proj',  # 注意：vLLM 合并 QKV
            'decoder.layers.0.mlp.dense_h_to_4h': 'model.layers.0.mlp.gate_up_proj',  # 合并 gate + up
            ...
        }
        """
        self.layer_name_mapping = layer_name_mapping
    
    def _shard_for_vllm(self, gathered_tensor, megatron_layer_name):
        """
        将 Megatron 格式的张量转换为 vLLM 格式
        """
        vllm_layer_name = self.layer_name_mapping[megatron_layer_name]
        
        # 情况 1: 简单重分片 (TP 份数变化)
        if 'self_attention.query' in megatron_layer_name:
            # Megatron: Q, K, V 分开存储，各 8 份
            # vLLM: QKV 合并存储，共 4 份
            q_shard = gathered_tensor  # [64, hidden, hidden/8]
            # 需要与 K, V 拼接后重新切分...
            
        # 情况 2: 结构重组 (如 MLP 的 gate + up 合并)
        elif 'mlp.dense_h_to_4h' in megatron_layer_name:
            # Megatron: gate 和 up 是两个独立矩阵
            # vLLM: 合并为 gate_up_proj [2 * intermediate_size, hidden]
            gate = gathered_tensor[:intermediate_size]
            up   = gathered_tensor[intermediate_size:]
            merged = torch.cat([gate, up], dim=0)
            # 再按 vLLM 的 TP 方式切分...
            
        return resharded_tensor

## 阶段 3：加载到 vLLM 引擎

In [ ]:
# vLLM 0.8.2+ 支持 per-tensor weight loading
self.inference_engine.load_weights([
    (vllm_layer_name, resharded_shard_for_this_tp_rank)
])

# 完整的 NCCL 重分片代码逻辑



In [ ]:
class MegatronVLLMShardingManager:
    """
    Context Manager: 进入时 Megatron → vLLM, 退出时 vLLM → Megatron
    """
    
    def __init__(self, 
                 module,                    # Megatron 模型 (AllGatherPPModel)
                 inference_engine,          # vLLM LLMEngine
                 model_config,
                 layer_name_mapping):
        self.module = module
        self.inference_engine = inference_engine
        self.model_config = model_config
        self.layer_name_mapping = layer_name_mapping
        
        # 获取 micro DP 通信组
        self.micro_dp_group = self._get_micro_dp_group()
        
        # 缓存原始 Megatron 分片，用于恢复
        self.original_shards = {}
    
    def _get_micro_dp_group(self):
        """
        构建 micro DP 组:
        同 DP rank、同 TP rank、跨 PP Stage 的 GPU
        """
        tp_rank = mpu.get_tensor_model_parallel_rank()
        dp_rank = mpu.get_data_parallel_rank()
        pp_size = mpu.get_pipeline_model_parallel_world_size()
        
        # 收集所有同 TP rank + 同 DP rank 的 GPU
        world_size = torch.distributed.get_world_size()
        micro_dp_ranks = []
        for rank in range(world_size):
            # 通过计算确定每个 rank 的 TP/PP/DP 位置
            rank_tp = rank % tp_size
            rank_pp = (rank // tp_size) % pp_size
            rank_dp = rank // (tp_size * pp_size)
            
            if rank_tp == tp_rank and rank_dp == dp_rank:
                micro_dp_ranks.append(rank)
        
        return dist.new_group(ranks=micro_dp_ranks)
    
    def _all_gather_micro_dp(self, local_param):
        """
        在 micro DP 组内 all_gather 参数
        聚合 PP 维度的分片
        """
        # 计算聚合后的大小
        pp_size = mpu.get_pipeline_model_parallel_world_size()
        gathered_shape = list(local_param.shape)
        # 假设第 0 维是层维度，按 PP 拼接
        gathered_shape[0] *= pp_size
        
        gathered = torch.empty(gathered_shape, 
                               dtype=local_param.dtype,
                               device=local_param.device)
        
        # NCCL all_gather
        dist.all_gather_into_tensor(gathered, local_param, 
                                    group=self.micro_dp_group)
        return gathered
    
    def _convert_tp_sharding(self, gathered, megatron_name, target_tp_size):
        """
        将 Megatron 的 TP 分片转换为 vLLM 的 TP 分片
        """
        current_tp_size = mpu.get_tensor_model_parallel_world_size()
        
        if current_tp_size == target_tp_size:
            # TP 大小相同，直接返回当前 rank 对应的分片
            tp_rank = mpu.get_tensor_model_parallel_rank()
            chunk_size = gathered.shape[-1] // current_tp_size
            return gathered[..., tp_rank*chunk_size:(tp_rank+1)*chunk_size]
        
        elif current_tp_size > target_tp_size:
            # Megatron TP 更大，需要合并分片
            # 例如 TP8 → TP4: 每 2 个 Megatron TP rank 合并为 1 个 vLLM TP rank
            merge_factor = current_tp_size // target_tp_size
            tp_rank = mpu.get_tensor_model_parallel_rank()
            vllm_tp_rank = tp_rank // merge_factor
            
            # 收集需要合并的分片 (在 TP 组内做 all_gather)
            merged = self._gather_and_merge(gathered, merge_factor)
            return merged
        
        else:
            # Megatron TP 更小，需要拆分分片 (较少见)
            split_factor = target_tp_size // current_tp_size
            ...
    
    def __enter__(self):
        """Megatron → vLLM 重分片"""
        weights_to_load = []
        
        for megatron_name, param in self.module.named_parameters():
            # 1. micro DP 组内 all_gather (聚合 PP 维度)
            gathered = self._all_gather_micro_dp(param.data)
            
            # 2. 缓存原始分片 (用于退出时恢复)
            self.original_shards[megatron_name] = param.data.clone()
            
            # 3. TP 维度重分片
            target_tp = self.inference_engine.tensor_parallel_size
            vllm_shard = self._convert_tp_sharding(gathered, megatron_name, target_tp)
            
            # 4. 层名映射
            vllm_name = self.layer_name_mapping.get(megatron_name, megatron_name)
            
            weights_to_load.append((vllm_name, vllm_shard))
        
        # 5. 批量加载到 vLLM
        self.inference_engine.load_weights(weights_to_load)
        
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        """vLLM → Megatron 恢复"""
        # 释放 vLLM 权重缓存
        self.inference_engine.reset_prefix_cache()
        
        # 恢复 Megatron 原始分片 (从 CPU 或重新 all_gather)
        for name, original_shard in self.original_shards.items():
            param = dict(self.module.named_parameters())[name]
            param.data.copy_(original_shard)
        
        # 清理
        self.original_shards.clear()
        torch.cuda.empty_cache()
        
        return False

# 关键优化点
##  通信量分析

In [ ]:
传统方案 (全局 AllGather):
┌────────────────────────────────────────────┐
│  所有 64 GPU 参与 AllGather                  │
│  通信量: 64 × 140GB = 8.96 TB (算法复杂度)    │
│  实际传输: 每个 GPU 发送 140GB, 接收 140GB    │
│  总带宽: 8960 GB                            │
└────────────────────────────────────────────┘

verl micro DP 方案:
┌────────────────────────────────────────────┐
│  16 个 micro DP 组，每组 4 GPU               │
│  每组内 all_gather 通信量: 4 × 35GB = 140 GB  │
│  (35GB = 140GB / 4 PP Stage)                 │
│  总通信量: 16 × 140 GB = 2240 GB            │
│  相比传统方案: 75% 通信量减少!                 │
└────────────────────────────────────────────┘

## 与 CUDA IPC 的结合

In [ ]:
对于 Actor 和 Rollout 在同一节点内的场景:
┌────────────────────────────────────────────┐
│  使用 CUDA IPC (Inter-Process Communication) │
│                                             │
│  GPU 0 (Megatron Actor)                     │
│  ┌─────────────────┐                        │
│  │  权重缓冲区 (显存) │ ←── CUDA IPC ──→     │
│  └─────────────────┘      (零拷贝共享)      │
│         ↑                                   │
│  GPU 1 (vLLM Rollout)                       │
│  ┌─────────────────┐                        │
│  │  直接访问同一显存  │                        │
│  │  (无需通过 CPU)   │                        │
│  └─────────────────┘                        │
│                                             │
│  节省: CPU↔GPU 拷贝时间 + 内存双份存储         │
└────────────────────────────────────────────┘

## 异步流水线

In [ ]:
时间轴:
t0: 开始重分片
  ├── t0~t1: micro DP all_gather (通信)
  ├── t1~t2: TP 重分片计算 (计算)
  ├── t2~t3: 加载到 vLLM (内存拷贝)
  │
  └── 同时: 将 Megatron 训练权重异步 offload 到 CPU
      (与上述操作并行，隐藏延迟)

t3: 生成阶段开始

# 总结

| 维度           | 传统方案              | verl NCCL 重分片                            |
| ------------ | ----------------- | ---------------------------------------- |
| **通信范围**     | 全局所有 GPU          | micro DP 组 (4 GPU)                       |
| **通信原语**     | `all_gather` (全局) | `all_gather` (局部) + `all_gather` (TP 组内) |
| **内存副本**     | 2 份完整模型           | 1 份活跃 + 分片缓存                             |
| **TP 转换**    | 不支持或全量重新切分        | 组内合并/拆分                                  |
| **层名映射**     | 手动处理              | `MegatronVLLMShardingManager` 自动处理       |
| **CUDA IPC** | 不支持               | 同节点零拷贝共享                                 |

## NCCL 重分片的核心设计哲学是 "局部性优先" 
-  利用 RL 训练中 Actor 模型训练与生成通常共享物理 GPU 的特点，通过 micro DP 组将全局通信转化为局部通信，配合原地内存转换，实现了近乎零开销的训练/生成模式切换。